In [1]:
pip install mlxtend --upgrade

  Using cached mlxtend-0.25.0-py3-none-any.whl.metadata (7.4 kB)
Using cached mlxtend-0.25.0-py3-none-any.whl (1.4 MB)
  Attempting uninstall: mlxtend
    Found existing installation: mlxtend 0.24.0
    Uninstalling mlxtend-0.24.0:
      Successfully uninstalled mlxtend-0.24.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [3]:
basket = pd.read_csv('customer_basket.csv')
clusters = pd.read_csv('cluster_assignments.csv')
basket = basket.merge(clusters, on='customer_id')

print(f"Total baskets: {len(basket)}")
print(f"Baskets per cluster:\n{basket['cluster'].value_counts().sort_index()}")

Total baskets: 100000
Baskets per cluster:
cluster
0    10185
1    31919
2    35952
3    14335
4     7609
Name: count, dtype: int64


In [4]:
basket['list_of_goods'] = basket['list_of_goods'].apply(ast.literal_eval)

# Quick sanity check
print(f"Sample basket: {basket['list_of_goods'].iloc[0]}")
print(f"Sample cluster: {basket['cluster'].iloc[0]}")

Sample basket: ['chicken', 'rice', 'pepper', 'whole wheat rice', 'shrimp', 'toothpaste', 'gums', 'cereals', 'bluetooth headphones', 'vacuum cleaner', 'body spray']
Sample cluster: 1


In [5]:
def get_association_rules(transactions, min_support=0.02, min_confidence=0.2):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df = pd.DataFrame(te_array, columns=te.columns_)
    frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        print("No frequent itemsets found")
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    return rules.sort_values('lift', ascending=False)

In [6]:
all_products = [item for sublist in basket['list_of_goods'] for item in sublist]
print(f"Unique products: {len(set(all_products))}")
print(f"Most common products:\n{pd.Series(all_products).value_counts().head(10)}")

Unique products: 164
Most common products:
asparagus       12811
airpods         12145
cereals          9952
fresh bread      9934
butter           9654
eggs             9241
protein bar      8695
cooking oil      8623
toilet paper     8395
babies food      8318
Name: count, dtype: int64


In [7]:
print(basket['cluster'].value_counts().sort_index())
print(basket['cluster'].isna().sum())

cluster
0    10185
1    31919
2    35952
3    14335
4     7609
Name: count, dtype: int64
0


In [8]:
print(basket[basket['cluster'].isna()].shape)
basket = basket.dropna(subset=['cluster'])
basket['cluster'] = basket['cluster'].astype(int)
print(basket['cluster'].value_counts().sort_index())

(0, 4)
cluster
0    10185
1    31919
2    35952
3    14335
4     7609
Name: count, dtype: int64


In [9]:
# 4. Run per cluster and store results
cluster_rules = {}
for cluster_id in sorted(basket['cluster'].unique()):
    transactions = basket[basket['cluster'] == cluster_id]['list_of_goods'].tolist()
    rules = get_association_rules(transactions)
    cluster_rules[cluster_id] = rules

In [10]:
for cluster_id in sorted(basket['cluster'].unique()):
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id} — Top Unique Rules by Lift")
    print(f"{'='*50}")
    rules = cluster_rules[cluster_id]
    if not rules.empty:
        # Convert frozensets to sorted tuples to identify duplicates
        rules = rules.copy()
        rules['pair'] = rules.apply(
            lambda row: tuple(sorted([
                str(sorted(row['antecedents'])), 
                str(sorted(row['consequents']))
            ])), axis=1
        )
        # Keep only one direction per pair
        unique_rules = rules.drop_duplicates(subset='pair')
        top10 = unique_rules.nlargest(10, 'lift')[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        display(top10)


CLUSTER 0 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
640,(green tea),"(cereals, eggs)",0.021600,0.197309,1.323845
638,"(green tea, eggs)",(cereals),0.021600,0.480349,1.321902
660,"(cereals, oil)",(eggs),0.022484,0.459839,1.307865
641,(eggs),"(cereals, green tea)",0.021600,0.061435,1.300871
678,"(cereals, tea)",(eggs),0.038783,0.456120,1.297287
600,"(oatmeal, butter)",(milk),0.020619,0.255164,1.291673
754,(oatmeal),"(cereals, milk)",0.021797,0.103690,1.287904
212,(green tea),(salt),0.020422,0.186547,1.283772
586,(butter),"(fresh bread, whole weat flour)",0.020815,0.058661,1.279357
659,(eggs),"(cereals, oatmeal)",0.038979,0.110863,1.278753



CLUSTER 1 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
24,(deodorant),(shower gel),0.025502,0.292175,3.560878
32,(shower gel),(shampoo),0.025408,0.309660,3.545209
38,(tooth brush),(shower gel),0.024813,0.278579,3.395174
23,(shampoo),(deodorant),0.025126,0.287661,3.295716
35,(shampoo),(tooth brush),0.024875,0.284792,3.197423
40,(toothpaste),(shower gel),0.024938,0.257106,3.133473
27,(tooth brush),(deodorant),0.024218,0.271896,3.115091
37,(shampoo),(toothpaste),0.025690,0.294118,3.032281
42,(toothpaste),(tooth brush),0.025909,0.267119,2.999004
28,(deodorant),(toothpaste),0.024625,0.282125,2.908639



CLUSTER 2 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
3,(energy drink),(airpods),0.023448,0.314435,2.669949
12,(cooking oil),(napkins),0.020138,0.232723,2.622013
1,(bluetooth headphones),(airpods),0.022141,0.300491,2.551545
4,(cooking oil),(babies food),0.021584,0.249437,2.484837
15,(dog food),(napkins),0.022586,0.218868,2.465917
9,(babies food),(napkins),0.021584,0.215018,2.422541
7,(babies food),(dog food),0.024255,0.241618,2.341417
11,(dog food),(cooking oil),0.020444,0.198113,2.289478



CLUSTER 3 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
51,(energy drink),(bluetooth headphones),0.026299,0.286257,3.060022
45,(salad),(avocado),0.023369,0.259489,2.997398
52,(carrots),(salad),0.026509,0.264993,2.942428
58,(salad),(spinach),0.025253,0.280403,2.753133
43,(carrots),(avocado),0.023788,0.237796,2.746826
0,(airpods),(bluetooth headphones),0.042065,0.249896,2.671338
33,(asparagus),(salad),0.040251,0.239518,2.659564
46,(avocado),(spinach),0.023160,0.267526,2.626704
6,(airpods),(energy drink),0.040460,0.240365,2.616270
18,(avocado),(asparagus),0.036972,0.427075,2.541353



CLUSTER 4 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
0,(airpods),(asparagus),0.021159,0.162298,1.036884


**Support** - proportion of baskets that contain both items

**Confidence** - percentage of bought together ('If someone buys item A, we have a x% confidence that buys item B)

**Lift** - measure of how much more likely someone is to buy two items together compared to independently
- Lift = 1 → no association, purely coincidental
- Lift > 1 → positive association, bought together more than expected
- Lift < 1 → negative association, rarely bought together

# Cluster Campaigns

**Cluster 0 - "Big Family Shopper"**
- eggs + green tea → cereals (lift 1.37), cereals + eggs → green tea (lift 1.36), honey → cereals + outmeal (lift 1.32) - Strong healthy breakfast pattern
- high drinks (alcohol and non-alcohol) and hygiene spending
- Campaign: **"Family Breakfast Bundle - buy cereals + green tea + eggs and get honey and outmeal 30% off"**, **"Party Ready Bundle - buy 3 alcoholic drinks and get 20% off non-alcoholic drinks"** and **"Family Hygiene Pack - buy €30 in hygiene products and get the cheapest one free"**

**Cluster 1 - "Bargain Hunter"**
- asparagus + carrots → salad (lift 3.56), salad → asparagus + avocado (lift 3.52), asparagus + carrots → avocado (lift 3.45), spinach → asparagus + carrots (lift 3.28), bluetooth headphones → energy drink (lift 3.27)
- spend mostly on sales
- Campaign: **"Healthy Plate Deal - buy asparagus + salad + spinach and get carrots + avocado at half price"** and **"Double Promotion Day — every Wednesday all your usual promoted items get an extra 10% off"**

**Cluster 2 - "Most Loyal"**
- energy drink → airpods (lift 2.61), bluetooth headphones → airpods (lift 2.49)
- high fresh_food_ratio, hygiene and tech spending 
- has_loyalty_card is 1
- Campaign: **"Tech Bundle - buy AirPods and get a free Energy Drink"** or **"Loyal Member Reward - use your loyalty card and get 10% off everything from the category Fresh Food/Hygiene/Tech"** (category changes every week)

**Cluster 3 - "New Customers + Tech Enthusiast"**
- Surprisingly dominated by hygiene products - shower gel → shampoo (lift 3.44), deodorant → shower gel (lift 3.44), tooth brush → shower gel (lift 3.33), shampoo → tooth brush (lift 3.19)
- high fresh_food_ratio and tech spending 
- Campaign: **"Hygiene Kit - buy shampoo + shower gel + deodorant and get a toothbrush free"**, **"Fresh Tech Deal - buy €50 in fresh food and get 10% off any electronics"** and **"Welcome to the Club - sign up for loyalty card and get 15% off in Hygiene/Tech products"** (category changes every week)

**Cluster 4 - "Biggest Buyers"**
- Only 1 meaningful rule (asparagus → airpods, lift 1.07) which is very weak
- Don't base the campaign on association rules here - use their spending profile instead
- Campaign: **"VIP Exclusive — spend over €150 and get 15% off your entire basket"**
